# 0. Init

In [0]:
import requests
import uuid
import json
from datetime import datetime, date, timedelta
import calendar
import os
from pyspark.sql.types import StructType, StructField, DoubleType, StringType,TimestampNTZType, TimestampType
from pyspark.sql import functions as F
from bronze_config import LOAD_CONFIG, INGESTION_CONFIG


In [0]:
weather_config=INGESTION_CONFIG['weather']

In [0]:
weather_config

# 1. Configuration


In [0]:
TGT_START_DATE = date(2024,12,31)
TGT_END_DATE = date(2025,12,31)

OPEN_METEO_URL= "https://archive-api.open-meteo.com/v1/archive"

TIME_ZONE="America/New_York"

LOCATIONS = [
    {
        "borough": "Manhattan",
        "latitude": 40.7831,
        "longitude": -73.9712
    },
    {
        "borough": "Brooklyn",
        "latitude": 40.6782,
        "longitude": -73.9442
    },
    {
        "borough": "Queens",
        "latitude": 40.7282,
        "longitude": -73.7949
    },
    {
        "borough": "Bronx",
        "latitude": 40.8448,
        "longitude": -73.8648
    },
    {
        "borough": "Staten Island",
        "latitude": 40.5795,
        "longitude": -74.1502
    }
]

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "snowfall",
    "snow_depth",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m"
]


# 2. Create control + Bronze tables


In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {weather_config['state_table']}(
        source_name STRING,
        entity_name STRING,
        last_ingested_date Date,
        last_run_id STRING,
        updated_at TIMESTAMP

    )
    USING DELTA
    """
)

In [0]:
spark.sql(f"""
          MERGE INTO {weather_config['state_table']} AS tgt
          USING (
              select "open_meteo" AS source_name,
              "weather_hourly" AS entity_name,
              CAST('2024-12-30' AS DATE) AS last_ingested_date
          ) AS src

          ON src.source_name=tgt.source_name 
          AND src.entity_name = tgt.entity_name

          WHEN NOT MATCHED THEN 
              INSERT (source_name, entity_name, last_ingested_date, last_run_id, updated_at)
              VALUES(src.source_name, src.entity_name, src.last_ingested_date, NULL, current_timestamp())
          """)

In [0]:
display(spark.table(weather_config['state_table']))

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {weather_config['table']}(
              borough STRING,
              timestamp timestamp,
              latitude DOUBLE,
              longitude DOUBLE,
              temperature DOUBLE,
              humidity DOUBLE,
              rain DOUBLE,
              snowfall DOUBLE,
              snow_depth DOUBLE,
              precipitation DOUBLE,
              wind_speed DOUBLE,
              wind_direction DOUBLE,
              source_name STRING,
              source_start_date DATE,
              source_end_date DATE,
              run_id STRING,
              source_file STRING,
              ingested_at TIMESTAMP
          )
          USING DELTA 
          """)

In [0]:
display(spark.table(weather_config['table']))

# 3. Read last watermark


In [0]:
state_row=spark.table(weather_config['state_table']).filter((F.col("source_name") == "open_meteo") & (F.col("entity_name") == "weather_hourly")).select("last_ingested_date").first()

last_ingested_date=state_row['last_ingested_date']

print(f"last_ingested_date: {last_ingested_date}")

# 4. Determine next month

In [0]:
if last_ingested_date >= TGT_END_DATE:
    print("All target data has already been ingested!")
else:
    next_start_date = last_ingested_date + timedelta(days=1)
    year= next_start_date.year
    month=next_start_date.month
    last_day=calendar.monthrange(year,month)[1]
    next_end_date= date(year, month, last_day)

    if next_end_date > TGT_END_DATE:
        next_end_date = TGT_END_DATE
    print(f"Next Ingestion: \n Start: {next_start_date}\n End: {next_end_date}")
    


# 5. Call API


In [0]:
latitude_param = ",".join(str(location["latitude"]) for location in LOCATIONS)

longitude_param = ",".join(str(location["longitude"]) for location in LOCATIONS)

params={
    "latitude": latitude_param,
    "longitude": longitude_param,

    "start_date": next_start_date.isoformat(),
    "end_date":next_end_date.isoformat(),

    "hourly":",".join(HOURLY_VARIABLES),

    "timezone": TIME_ZONE,

    "temperature_unit": "fahrenheit",
	"wind_speed_unit": "mph",
	"precipitation_unit": "inch",
}

In [0]:
response=requests.get(OPEN_METEO_URL, params, timeout=60)

print("HTTP STATUS", response.status_code)

response.raise_for_status()

payload=response.json()

print(type(payload))


# 6. Save raw JSON

In [0]:
run_id= str(uuid.uuid4())

print(run_id)

In [0]:
raw_month_path=(f"{LOAD_CONFIG['raw_path']}/"
                f"year={year}/"
                f"month={month}/"
                f"run_id={run_id}"
                )


In [0]:
os.makedirs(raw_month_path, exist_ok=True)

In [0]:
raw_file_path=f"{raw_month_path}/response.json"

In [0]:
with open(raw_file_path, "w") as f:
    json.dump(payload, f)

print(f"Raw response saved to {raw_file_path}")

# 7. Parse JSON → DataFrame

In [0]:
with open(raw_file_path, "r") as f:
    raw_data=json.load(f)

In [0]:
raw_data

In [0]:
def parse_open_meteo(payload, locations):

    rows=[]

    if len(payload) != len(locations):
        raise ValueError("Number of API responses doesn't match" 
                         "number of requested locations")
        
    for location, location_data in zip(locations, payload):
        hourly=location_data["hourly"]
        times=hourly["time"]
        temperatures=hourly["temperature_2m"]
        humidities=hourly["relative_humidity_2m"]
        rains=hourly["rain"]
        snowfalls=hourly["snowfall"]
        snow_depths=hourly['snow_depth']
        precipitations=hourly["precipitation"]
        wind_speeds=hourly["wind_speed_10m"]
        wind_directions=hourly["wind_direction_10m"]
        latitude=location_data['latitude']
        longitude=location_data['longitude']

        lengths={
            len(times), len(temperatures), len(rains), len(snowfalls), len(snow_depths), len(precipitations), len(wind_speeds), len(wind_directions)
        }

        if len(lengths) != 1:
            raise ValueError(f"Hourly arrays have different lengths"
                             f"for {location['borough']}")
            
        for i in range(len(times)):
            rows.append({
                "borough": location['borough'],
                "latitude": latitude,
                "longitude": longitude,
                "timestamp": times[i],
                "temperature": temperatures[i],
                "humidity": humidities[i],
                "rain": rains[i],
                "snowfall": snowfalls[i],
                "snow_depth": snow_depths[i],
                "precipitation": precipitations[i],
                "wind_speed": wind_speeds[i],
                "wind_direction": wind_directions[i]
            })

    return rows
           

In [0]:
rows= parse_open_meteo(raw_data, LOCATIONS)

print("Rows:", len(rows))

rows[:2]

In [0]:
weather_schema=StructType(
    [
        StructField("borough", StringType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("timestamp", StringType(), True),
        StructField("temperature", DoubleType(), True),
        StructField("humidity", DoubleType(), True),
        StructField("rain", DoubleType(), True),
        StructField("snowfall", DoubleType(), True),
        StructField("snow_depth", DoubleType(), True),
        StructField("precipitation", DoubleType(), True),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", DoubleType(), True)
        
    ]
)

In [0]:
weather_df= spark.createDataFrame(rows, schema=weather_schema)

In [0]:
weather_df = weather_df.withColumn(
    "timestamp",
    F.to_timestamp_ntz("timestamp")
)

In [0]:
display(weather_df.limit(10))

#8. Add metadata

In [0]:
weather_df= weather_df.withColumn("source_name", F.lit("open_meteo")).withColumn("source_start_date", F.lit(next_start_date.isoformat())).withColumn("source_end_date", F.lit(next_end_date.isoformat())).withColumn("run_id", F.lit(run_id)).withColumn("source_file", F.lit(raw_file_path)).withColumn("ingested_at", F.current_timestamp())

# 9. DF → Bronze Delta

In [0]:
weather_df.createOrReplaceTempView("weather_batch")

In [0]:
spark.sql(f"""
          MERGE INTO {weather_config['table']} AS tgt USING weather_batch AS src
          ON tgt.borough=src.borough
          AND tgt.timestamp= src.timestamp

          WHEN MATCHED THEN UPDATE SET
          tgt.latitude= src.latitude,
          tgt.longitude= src.longitude,
          tgt.temperature = src.temperature,
          tgt.humidity= src.humidity,
          tgt.rain= src.rain,
          tgt.snowfall= src.snowfall,
          tgt.snow_depth= src.snow_depth,
          tgt.precipitation= src.precipitation,
          tgt.wind_speed= src.wind_speed,
          tgt.wind_direction= src.wind_direction,
          tgt.source_name= src.source_name,
          tgt.source_start_date= src.source_start_date,
          tgt.source_end_date = src.source_end_date,
          tgt.run_id = src.run_id,
          tgt.source_file = src.source_file,
          tgt.ingested_at = src.ingested_at
          WHEN NOT MATCHED THEN INSERT
          (borough,
            latitude,
            longitude,
            timestamp,
            temperature,
            humidity,
            rain,
            snowfall,
            snow_depth,
            precipitation,
            wind_speed,
            wind_direction,
            source_name,
            source_start_date,
            source_end_date,
            run_id,
            source_file,
            ingested_at)

            VALUES 
            (src.borough,
            src.latitude,
            src.longitude,
            src.timestamp,
            src.temperature,
            src.humidity,
            src.rain,
            src.snowfall,
            src.snow_depth,
            src.precipitation,
            src.wind_speed,
            src.wind_direction,
            src.source_name,
            src.source_start_date,
            src.source_end_date,
            src.run_id,
            src.source_file,
            src.ingested_at
            )

          """)

# 10. Update watermark

In [0]:
spark.sql(f"""
          MERGE INTO {weather_config['state_table']} AS tgt 
          USING (
              SELECT 'open_meteo' AS source_name,
              'weather_hourly' AS entity_name,
              CAST('{next_end_date.isoformat()}' AS DATE) AS last_ingested_date,
              '{run_id}' AS last_run_id
          ) AS src
          ON tgt.source_name = src.source_name
          AND tgt.entity_name = src.entity_name

          WHEN MATCHED THEN UPDATE SET
          tgt.last_ingested_date = src.last_ingested_date,
          tgt.last_run_id = src.last_run_id,
          tgt.updated_at = current_timestamp()

          WHEN NOT MATCHED THEN INSERT 
          (source_name, entity_name, last_ingested_date, last_run_id, updated_at)
          VALUES 
          (src.source_name, src.entity_name, src.last_ingested_date, src.last_run_id, current_timestamp())
          """)

In [0]:
%sql
select * from nyc_taxi_project.bronze.weather_ingestion_state